# A6. The PM2.5 lead equation after the fire-file correction: every number the text uses about it

2026-10-20: the gridded hotspot file that built all fire columns was found to be spatially wrong in most years (daily totals
agreed with FIRMS, the location of the fires did not). All fire columns were rebuilt from a FIRMS VIIRS 375 m regeneration
(`Scripts/colombia/rebuild_fire_columns.py`), every selection and search was re-run, and the PM2.5 lead equation became
complexity 13 of the dimensionally constrained front, seed 0:

    PM2.5 = sqrt( a (F_E,-2 + F_N) ) + b CO / sqrt(u)

This notebook gives, for that equation: (1) what the correction did (old equation on old and on corrected fire data, correlation
of PM2.5 with the old and the new fire columns), (2) the seed stability of the constrained search, (3) ablations and add-one tests
(what each part contributes, whether TCWV, w700, precipitation, sensible heat flux, BLH, T or RH add anything), (4) the size of the fire
term, and (5) a scan of fire sector x lag (which direction and delay carry the signal).
Scoring is leave-one-year-out with constants refitted on the training years; pooled R2 and mean of per-year R2 are both reported.

In [1]:
# Setup
import sys
from pathlib import Path
HERE = Path.cwd().resolve()
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
import warnings
import numpy as np, pandas as pd
from scipy.optimize import curve_fit
from scipy.stats import spearmanr
from sklearn.metrics import r2_score
from style import ROOT, DATA_OUT
from termmap_utils import load_front, term_presence, term_summary
warnings.filterwarnings("ignore")
B = ROOT / "Data/processed/colombia/bogota"
OLD = ROOT / "Data/processed/colombia/_oldfire_backup"

In [2]:
# Settings (edit here)
LEAD_FRONT = B / "bogota_pm25_pysr_units_fire2_pareto_front_seed0.csv"
SEED_TABLE = B / "bogota_pm25_pysr_units_fire2_scored.csv"
LEAD_C = 13
SAVE = True

In [3]:
# Models (p0 = constants printed by PySR)
def lead(X, a, b):                        # complexity 13, seed 0
    co, u, fn, fe2 = X
    return np.sqrt(np.clip(a * (fe2 + fn), 0, None)) + b * co / np.sqrt(u)
LEAD_P0 = [0.10428376, 15.679531]
LEAD_F = ["co", "wind_speed_era5", "hotspots_sector_N", "hotspots_sector_E_lag2"]
def co_only(X, b): return b * X[0]                                           # complexity 3
def co_wind(X, a): return a * np.sqrt(X[0] / X[1])                           # complexity 6 (fire-free)
def no_wind(X, a, b):                                                         # same two fire sectors, CO not divided by wind
    co, u, fn, fe2 = X; return np.sqrt(np.clip(a * (fe2 + fn), 0, None)) + b * co
def e2_only(X, a, b):
    co, u, fn, fe2 = X; return np.sqrt(np.clip(a * fe2, 0, None)) + b * co / np.sqrt(u)
def n_only(X, a, b):
    co, u, fn, fe2 = X; return np.sqrt(np.clip(a * fn, 0, None)) + b * co / np.sqrt(u)
def old_eq1(X, a, b, c, d):               # the former Eq. 1 (complexity 18 on the old fire data)
    co, w700, u, tcwv, hot = X
    return (co + (a - w700) * b / (u * tcwv)) * (c * np.sqrt(np.clip(hot, 0, None)) + d)
OLD_P0 = [0.29734862, 16.203215, 0.0617, 15.79]; OLD_F = ["co", "w700", "wind_speed_era5", "tcwv_lag1", "hotspots_sector_E_lag3"]

def load(p):
    d = pd.read_parquet(p); d["year"] = pd.to_datetime(d["date"]).dt.year; return d

def loyo(model, X, y, yrs, p0):
    X = tuple(np.asarray(a, float) for a in X)
    pf, _ = curve_fit(model, X, y, p0=p0, maxfev=50000)
    pred = np.full(len(y), np.nan)
    for yr in np.unique(yrs):
        te = yrs == yr
        try: p, _ = curve_fit(model, tuple(a[~te] for a in X), y[~te], p0=pf, maxfev=50000)
        except RuntimeError: p = pf
        pred[te] = model(tuple(a[te] for a in X), *p)
    per = [r2_score(y[yrs == yr], pred[yrs == yr]) for yr in np.unique(yrs)]
    return dict(pooled=r2_score(y, pred), mean_of_years=float(np.mean(per)), in_sample=r2_score(y, model(X, *pf)), n=len(y), constants=pf, pred=pred, per_year=per)

In [4]:
# 1. What the correction did
new = load(B / "bogota_master_dataset.parquet"); old = load(OLD / "bogota_master_dataset.parquet")
rows = []
dn = new.dropna(subset=LEAD_F + ["pm25"]).reset_index(drop=True)
r = loyo(lead, tuple(dn[c] for c in LEAD_F), dn.pm25.to_numpy(float), dn.year.to_numpy(), LEAD_P0)
rows.append(dict(model="new lead (c13), corrected fire data", **{k: r[k] for k in ("n", "pooled", "mean_of_years", "in_sample")}))
dno = old.dropna(subset=OLD_F + ["pm25"]).reset_index(drop=True)
r_old = loyo(old_eq1, tuple(dno[c] for c in OLD_F), dno.pm25.to_numpy(float), dno.year.to_numpy(), OLD_P0)
rows.append(dict(model="former Eq. 1 (c18), old fire data", **{k: r_old[k] for k in ("n", "pooled", "mean_of_years", "in_sample")}))
dnn = new.dropna(subset=OLD_F + ["pm25"]).reset_index(drop=True)
r_oc = loyo(old_eq1, tuple(dnn[c] for c in OLD_F), dnn.pm25.to_numpy(float), dnn.year.to_numpy(), OLD_P0)
rows.append(dict(model="former Eq. 1 (c18), corrected fire data", **{k: r_oc[k] for k in ("n", "pooled", "mean_of_years", "in_sample")}))
# fire-free skeleton on both data sets (the fire columns are the only thing that differs)
for lab, dd in (("corrected", dn), ("old", old.dropna(subset=["co", "wind_speed_era5", "pm25"]))):
    rr = loyo(co_wind, (dd.co, dd.wind_speed_era5), dd.pm25.to_numpy(float), dd.year.to_numpy(), [20.6])
    rows.append(dict(model=f"fire-free CO/wind skeleton, {lab} rows", n=rr["n"], pooled=rr["pooled"], mean_of_years=rr["mean_of_years"], in_sample=rr["in_sample"]))
what = pd.DataFrame(rows); print(what.round(3).to_string(index=False))
if SAVE: what.to_csv(DATA_OUT / "pm25_fire_correction_effect.csv", index=False)

# raw association of PM2.5 with the fire columns, old versus corrected file
cmp = []
for col in ["hotspots_sector_E_lag3", "hotspots_sector_E_lag2", "hotspots_sector_N", "hotspots_sector_NE_lag2", "hotspots_sector_E", "hotspots_sector_NE", "hotspots_sector_N_lag2"]:
    for lab, dd in (("old file", old), ("corrected file", new)):
        s = dd[[col, "pm25"]].dropna()
        cmp.append(dict(column=col, file=lab, n=len(s), pearson=float(s.corr().iloc[0, 1]), spearman=float(spearmanr(s[col], s.pm25)[0])))
cmp = pd.DataFrame(cmp); print(cmp.round(3).to_string(index=False))
if SAVE: cmp.to_csv(DATA_OUT / "pm25_fire_corr_old_vs_new.csv", index=False)

                                     model    n  pooled  mean_of_years  in_sample
       new lead (c13), corrected fire data 3283   0.677          0.655      0.685
         former Eq. 1 (c18), old fire data 3282   0.584          0.567      0.603
   former Eq. 1 (c18), corrected fire data 3282   0.665          0.645      0.680
fire-free CO/wind skeleton, corrected rows 3283   0.465          0.439      0.478
      fire-free CO/wind skeleton, old rows 3285   0.465          0.440      0.478
                 column           file    n  pearson  spearman
 hotspots_sector_E_lag3       old file 3652    0.218     0.386
 hotspots_sector_E_lag3 corrected file 4380    0.503     0.517
 hotspots_sector_E_lag2       old file 3652    0.216     0.388
 hotspots_sector_E_lag2 corrected file 4381    0.509     0.525
      hotspots_sector_N       old file 3652    0.133     0.295
      hotspots_sector_N corrected file 4383    0.402     0.246
hotspots_sector_NE_lag2       old file 3652    0.245     0.380
hots

In [5]:
# 2. Seed stability of the constrained search (scored fronts from 07_units_fire2_equation_discovery)
sc = pd.read_csv(SEED_TABLE); sc = sc[sc.seed.astype(str).isin(["0", "1", "2"])].copy(); sc["seed"] = sc.seed.astype(int)   # drop the rescored old-Eq.1 rows
seed_rows = []
for seed, g in sc.groupby("seed"):
    for lab, gg in (("complexity <= 16", g[g.complexity <= 16]), ("any complexity", g)):
        bb = gg.loc[gg.r2_pooled.idxmax()]
        seed_rows.append(dict(seed=int(seed), which=lab, complexity=int(bb.complexity), r2_pooled=bb.r2_pooled, r2_mean_of_years=bb.r2_mean_years, equation=bb.equation))
seeds = pd.DataFrame(seed_rows); print(seeds.drop(columns="equation").round(3).to_string(index=False))
# where CO, wind, fire (any sector), w700, sensible heat flux enter each seed's front
pres = []
for seed in sorted(sc.seed.unique()):
    fr = pd.read_csv(B / f"bogota_pm25_pysr_units_fire2_pareto_front_seed{int(seed)}.csv")[["complexity", "equation"]]
    mat = term_presence(fr)
    groups = {"CO": [v for v in mat.index if v == "co"], "wind speed": [v for v in mat.index if v == "wind_speed_era5"],
              "fire (any sector)": [v for v in mat.index if v.startswith("hotspots")], "w700": [v for v in mat.index if v == "w700"],
              "sensible heat flux": [v for v in mat.index if v.startswith("sensible")], "BLH": [v for v in mat.index if v == "blh_mean"]}
    for g, vs in groups.items():
        if not vs: pres.append(dict(seed=int(seed), term=g, first_complexity=np.nan, frac_present_from_first=0.0, present_at_top=False)); continue
        any_ = mat.loc[vs].max()
        idx = np.nonzero(any_.to_numpy())[0]
        if len(idx) == 0: pres.append(dict(seed=int(seed), term=g, first_complexity=np.nan, frac_present_from_first=0.0, present_at_top=False)); continue
        span = any_.to_numpy()[idx[0]:]
        pres.append(dict(seed=int(seed), term=g, first_complexity=int(mat.columns[idx[0]]), frac_present_from_first=float(span.mean()), present_at_top=bool(any_.iloc[-1])))
pres = pd.DataFrame(pres); print(pres.to_string(index=False))
if SAVE: seeds.to_csv(DATA_OUT / "pm25_constrained_seeds.csv", index=False); pres.to_csv(DATA_OUT / "pm25_constrained_term_presence_by_seed.csv", index=False)

 seed            which  complexity  r2_pooled  r2_mean_of_years
    0 complexity <= 16          13      0.677             0.655
    0   any complexity          18      0.693             0.676
    1 complexity <= 16          16      0.669             0.649
    1   any complexity          20      0.685             0.665
    2 complexity <= 16          16      0.692             0.676
    2   any complexity          24      0.704             0.688
 seed               term  first_complexity  frac_present_from_first  present_at_top
    0                 CO               3.0                 1.000000            True
    0         wind speed               6.0                 0.833333            True
    0  fire (any sector)               7.0                 1.000000            True
    0               w700               NaN                 0.000000           False
    0 sensible heat flux              16.0                 1.000000            True
    0                BLH               NaN      

In [6]:
# 3. Ablations and add-one tests, all on the same rows and the same protocol
y = dn.pm25.to_numpy(float); yrs = dn.year.to_numpy(); XL = tuple(dn[c] for c in LEAD_F)
abl = []
def add(name, model, X, p0):
    rr = loyo(model, X, y, yrs, p0); abl.append(dict(variant=name, pooled=rr["pooled"], mean_of_years=rr["mean_of_years"], in_sample=rr["in_sample"], constants=", ".join(f"{v:.4g}" for v in rr["constants"]))); return rr
rl = add("lead (c13)", lead, XL, LEAD_P0)
add("CO only (c3)", co_only, XL, [23.9])
add("CO and wind, no fire (c6)", co_wind, XL, [20.6])
add("fire from both sectors, CO not divided by wind", no_wind, XL, [0.1, 17.0])
add("east sector lag 2 only", e2_only, XL, [0.1, 15.7])
add("north sector only", n_only, XL, [0.1, 15.7])
# add-one multiplicative factor (1 + k z) on the whole equation, z = standardised candidate
extra = ["tcwv_lag1", "tcwv", "w700", "precip_total_mm", "precip_total_mm_lag1", "sensible_heat_flux", "blh_mean", "temp_2m", "rh_2m", "radiation_mean"]
extra = [e for e in extra if e in dn.columns]
dx = new.dropna(subset=LEAD_F + ["pm25"] + extra).reset_index(drop=True)
print("rows for the add-one tests:", len(dx), "of", len(dn))
yx = dx.pm25.to_numpy(float); yrx = dx.year.to_numpy(); XLx = tuple(dx[c] for c in LEAD_F)
base_x = loyo(lead, XLx, yx, yrx, LEAD_P0)
addone = []
for e in extra:
    z = ((dx[e] - dx[e].mean()) / dx[e].std()).to_numpy(float)
    m = lambda X, a, b, k, _z=None: lead(X[:4], a, b) * (1 + k * X[4])
    rr = loyo(m, XLx + (z,), yx, yrx, LEAD_P0 + [0.0])
    addone.append(dict(added_factor=e, pooled=rr["pooled"], gain_pooled=rr["pooled"] - base_x["pooled"], mean_of_years=rr["mean_of_years"], gain_mean=rr["mean_of_years"] - base_x["mean_of_years"],
                       k_per_sd=float(rr["constants"][2]), n=rr["n"]))
addone = pd.DataFrame(addone); print(pd.DataFrame(abl).round(3).to_string(index=False)); print("base on the add-one rows:", round(base_x["pooled"], 3)); print(addone.round(4).to_string(index=False))
if SAVE: pd.DataFrame(abl).to_csv(DATA_OUT / "pm25_lead_ablations.csv", index=False); addone.to_csv(DATA_OUT / "pm25_lead_addone.csv", index=False)

rows for the add-one tests: 2917 of 3283


                                       variant  pooled  mean_of_years  in_sample      constants
                                    lead (c13)   0.677          0.655      0.685 0.09794, 16.13
                                  CO only (c3)   0.370          0.341      0.381          23.85
                     CO and wind, no fire (c6)   0.465          0.439      0.478          20.62
fire from both sectors, CO not divided by wind   0.585          0.555      0.597  0.0928, 17.38
                        east sector lag 2 only   0.651          0.628      0.660  0.1198, 16.56
                             north sector only   0.620          0.593      0.630   0.1968, 18.1


base on the add-one rows: 0.668
        added_factor  pooled  gain_pooled  mean_of_years  gain_mean  k_per_sd    n
           tcwv_lag1  0.6795       0.0113         0.6536     0.0124   -0.0441 2917
                tcwv  0.6796       0.0114         0.6542     0.0130   -0.0477 2917
                w700  0.6709       0.0027         0.6437     0.0025   -0.0451 2917
     precip_total_mm  0.6674      -0.0008         0.6415     0.0003   -0.0144 2917
precip_total_mm_lag1  0.6678      -0.0004         0.6418     0.0006   -0.0154 2917
  sensible_heat_flux  0.6903       0.0221         0.6680     0.0268   -0.0660 2917
            blh_mean  0.6645      -0.0037         0.6389    -0.0023    0.0244 2917
             temp_2m  0.6681      -0.0000         0.6414     0.0002    0.0160 2917
               rh_2m  0.6657      -0.0025         0.6382    -0.0030   -0.0115 2917
      radiation_mean  0.6870       0.0189         0.6629     0.0216    0.0599 2917


In [7]:
# 4. Size of the terms and of the fire effect (full-data constants)
a_, b_ = rl["constants"]
Ft = np.sqrt(np.clip(a_ * (dn.hotspots_sector_E_lag2 + dn.hotspots_sector_N), 0, None)); Ct = b_ * dn.co / np.sqrt(dn.wind_speed_era5)
tot = Ft + Ct; share = Ft / tot
fire_sum = dn.hotspots_sector_E_lag2 + dn.hotspots_sector_N
eff = dict(a=a_, b=b_, mean_pm25=float(dn.pm25.mean()), mean_fire_term=float(Ft.mean()), mean_co_term=float(Ct.mean()),
           fire_share_of_prediction_median=float(share.median()), fire_share_p90=float(share.quantile(0.9)), fire_share_max=float(share.max()),
           fire_term_ugm3_median=float(Ft.median()), fire_term_ugm3_p90=float(Ft.quantile(0.9)), fire_term_ugm3_max=float(Ft.max()),
           fire_count_median=float(fire_sum.median()), fire_count_p90=float(fire_sum.quantile(0.9)), fire_count_max=float(fire_sum.max()),
           share_variance_fire=float(np.cov(Ft, tot)[0, 1] / np.var(tot)), share_variance_co_term=float(np.cov(Ct, tot)[0, 1] / np.var(tot)),
           days_fire_zero=float((fire_sum == 0).mean()),
           corr_fire_term_co_term=float(np.corrcoef(Ft, Ct)[0, 1]))
# CO per ug/m3 at calm vs windy days (the slope b / sqrt(u))
eff["slope_ugm3_per_ppm_at_median_wind"] = float(b_ / np.sqrt(dn.wind_speed_era5.median()))
eff["wind_p10"], eff["wind_p50"], eff["wind_p90"] = [float(dn.wind_speed_era5.quantile(q)) for q in (0.1, 0.5, 0.9)]
eff["slope_at_wind_p10"], eff["slope_at_wind_p90"] = float(b_ / np.sqrt(eff["wind_p10"])), float(b_ / np.sqrt(eff["wind_p90"]))
# monthly mean of the fire term (does it follow the Orinoco dry season?)
mon = pd.DataFrame(dict(month=pd.to_datetime(dn["date"]).dt.month, fire_term=Ft, co_term=Ct, pm25=dn.pm25)).groupby("month").mean()
print({k: round(v, 3) for k, v in eff.items()}); print(mon.round(2).T.to_string())
if SAVE: pd.DataFrame([eff]).to_csv(DATA_OUT / "pm25_lead_term_sizes.csv", index=False); mon.to_csv(DATA_OUT / "pm25_lead_monthly_terms.csv")

{'a': 0.098, 'b': 16.131, 'mean_pm25': 18.294, 'mean_fire_term': 5.374, 'mean_co_term': 12.87, 'fire_share_of_prediction_median': 0.271, 'fire_share_p90': 0.48, 'fire_share_max': 0.791, 'fire_term_ugm3_median': 4.056, 'fire_term_ugm3_p90': 10.899, 'fire_term_ugm3_max': 23.785, 'fire_count_median': 168.0, 'fire_count_p90': 1212.8, 'fire_count_max': 5776.0, 'share_variance_fire': 0.356, 'share_variance_co_term': 0.644, 'days_fire_zero': 0.004, 'corr_fire_term_co_term': 0.14, 'slope_ugm3_per_ppm_at_median_wind': 16.433, 'wind_p10': 0.63, 'wind_p50': 0.964, 'wind_p90': 1.658, 'slope_at_wind_p10': 20.33, 'slope_at_wind_p90': 12.527}
month         1      2      3      4      5      6      7      8      9      10     11     12
fire_term   7.81  10.29  11.40   7.50   3.74   2.32   3.09   3.33   3.22   3.38   3.53   5.19
co_term    11.85  14.01  15.69  13.46  12.13   9.88   8.98   9.99  12.44  14.66  16.54  14.92
pm25       20.21  24.96  26.93  18.94  15.21  12.95  11.33  14.06  16.46  18.40  2

In [8]:
# 5. Fire sector x lag scan: which direction and delay carry the signal beyond CO and ventilation?
SECT = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
def scan(df, label):
    d = df.dropna(subset=["co", "wind_speed_era5", "pm25"]).reset_index(drop=True)
    b0 = float(np.sum(d.pm25 * d.co / np.sqrt(d.wind_speed_era5)) / np.sum((d.co / np.sqrt(d.wind_speed_era5)) ** 2))
    res = d.pm25 - b0 * d.co / np.sqrt(d.wind_speed_era5)
    out = []
    for s in SECT:
        for k in (0, 1, 2, 3):
            col = f"hotspots_sector_{s}" + (f"_lag{k}" if k else "")
            if col not in d: continue
            m = d[col].notna()
            out.append(dict(file=label, sector=s, lag=k, spearman_with_residual=float(spearmanr(d.loc[m, col], res[m])[0]), n=int(m.sum())))
    return pd.DataFrame(out)
scn = pd.concat([scan(new, "corrected"), scan(old, "old")])
piv = scn.pivot_table(index=["file", "sector"], columns="lag", values="spearman_with_residual").reindex(SECT, level=1)
print(piv.round(3).to_string())
if SAVE: scn.to_csv(DATA_OUT / "pm25_fire_sector_lag_scan.csv", index=False)

lag                   0      1      2      3
file      sector                            
corrected N       0.470  0.429  0.386  0.357
          NE      0.430  0.430  0.421  0.402
          E       0.418  0.418  0.413  0.399
          SE      0.294  0.297  0.286  0.252
          S       0.105  0.141  0.136  0.106
          SW      0.024  0.027  0.014 -0.023
          W       0.331  0.287  0.241  0.190
          NW      0.341  0.305  0.278  0.261
old       N       0.175  0.185  0.202  0.191
          NE      0.171  0.190  0.203  0.197
          E       0.195  0.205  0.219  0.213
          SE      0.122  0.138  0.155  0.147
          S       0.066  0.075  0.081  0.079
          SW     -0.008  0.005  0.001 -0.011
          W       0.027  0.026  0.046  0.030
          NW      0.116  0.117  0.121  0.108
